# OPERA Sentinel-1 RTC backscatter (NASA Earthdata)

Download OPERA Radiometric-Terrain-Corrected Sentinel-1 backscatter from the ASF DAAC over a small area and map the VV channel in decibels — the kind of SAR layer used for flood and land-surface monitoring. Cloud-Optimized GeoTIFF **raster**, read with `pyramids`. Live query — needs the `[earthdata]` extra and EDL credentials, **and the ASF application authorized for your EDL account** (see [Authentication](../../reference/earthdata/authentication.md)); wrapped for nbval-lax safety offline.

> **ASF auth note:** ASF's datapool uses an EDL OAuth redirect that drops a bearer **token** across hosts (HTTP 401). ASF downloads therefore need **username/password** (or a `~/.netrc` entry) so `earthaccess` can hold the session — a bare `EARTHDATA_TOKEN` is not enough — or in-region S3. You must also authorize the *Alaska Satellite Facility Data Access* application for your EDL account (see Authentication).

In [ ]:
from pathlib import Path

from earthlens import EarthLens

OUT_DIR = Path('earthdata_output')
OUT_DIR.mkdir(exist_ok=True)

In [ ]:
paths = None
try:
    paths = EarthLens(
        data_source='earthdata',
        dataset='OPERA_L2_RTC-S1_V1', variables=['VV'],
        start='2024-01-01',
        end='2024-01-13',
        aoi=[-121.8, 36.3, -121.5, 36.6],
        path=str(OUT_DIR),
    ).download(progress_bar=False)
    print(len(paths), 'file(s):', [Path(p).name for p in paths][:4])
except Exception as exc:
    print(f'skipped live query: {type(exc).__name__}: {exc}')

In [ ]:
if paths:
    try:
        import numpy as np
        import matplotlib.pyplot as plt
        from pyramids.dataset import Dataset

        vv = next(p for p in paths if str(p).upper().endswith('_VV.TIF'))
        arr = Dataset.read_file(str(vv)).read_array().astype('float32')
        arr[arr <= 0] = np.nan
        db = 10.0 * np.log10(arr)
        fig, ax = plt.subplots(figsize=(8, 7))
        im = ax.imshow(db, cmap='gray', vmin=-25, vmax=0)
        fig.colorbar(im, ax=ax, label='VV backscatter (dB)')
        ax.set_title('OPERA RTC-S1 VV backscatter')
        plt.tight_layout()
        plt.show()
    except Exception as exc:
        print(f'read/plot step skipped: {type(exc).__name__}: {exc}')